# 01 – Exploratory Data Analysis: Labeled Faces in the Wild (LFW)

**Goal:** understand the dataset before building any models.

- How many people and images are there?
- Is the dataset balanced across people?
- What do the images look like (size, colour, lighting)?

The first run downloads LFW (~200 MB) to `~/scikit_learn_data`. Later runs load it from that cache.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.datasets import fetch_lfw_people

sns.set_theme(style="whitegrid")
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

## 1. Load the data

`min_faces_per_person=20` keeps only people with at least 20 images, so every class has enough examples to train and test on.

In [ ]:
MIN_FACES = 20

lfw = fetch_lfw_people(min_faces_per_person=MIN_FACES, resize=1.0, color=True)
images, y, names = lfw.images, lfw.target, lfw.target_names

print(f"Images:       {images.shape[0]}")
print(f"Image shape:  {images.shape[1:]}  (height, width, RGB)")
print(f"Pixel range:  {images.min():.2f} – {images.max():.2f}")
print(f"People:       {len(names)}")

## 2. Class balance

Face datasets are often very unbalanced: a few famous people have hundreds of photos, most have only a few. This matters for evaluation, because plain accuracy rewards a model that always guesses the biggest class. That's why we'll also report **macro F1** later.

In [ ]:
counts = (pd.Series(names[y]).value_counts()
            .rename_axis("person").reset_index(name="n_images"))

print(counts["n_images"].describe().round(1))
print(f"\nLargest class share: {counts['n_images'].iloc[0] / len(y):.1%}")
counts.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
sns.barplot(data=counts, y="person", x="n_images", color="steelblue", ax=ax)
ax.set_title("Images per person (LFW, ≥20 images)")
ax.set_xlabel("Number of images")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 3. Sample images

One random image per person for the first 18 people. Look for variation in pose, lighting, expression and occlusion (glasses, hands), since that's what makes recognition hard.

In [ ]:
rng = np.random.default_rng(42)
fig, axes = plt.subplots(3, 6, figsize=(14, 8))

for ax, person_id in zip(axes.flat, range(len(names))):
    idx = rng.choice(np.flatnonzero(y == person_id))
    ax.imshow(images[idx])
    ax.set_title(names[person_id].split()[-1], fontsize=9)
    ax.axis("off")

plt.suptitle("One random image per person")
plt.tight_layout()
plt.show()

In [ ]:
# Variation within one person: the same identity can look very different
person_id = int(np.flatnonzero(names == counts["person"].iloc[0])[0])  # person with most images
idx = rng.choice(np.flatnonzero(y == person_id), size=8, replace=False)

fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for ax, i in zip(axes, idx):
    ax.imshow(images[i])
    ax.axis("off")
plt.suptitle(f"8 images of {names[person_id]}")
plt.tight_layout()
plt.show()

## 4. Brightness and the "average face"

Brightness differences are one source of variation the model has to be robust to. The mean face shows how well the images are aligned: a sharp mean face means faces sit in roughly the same position in every image.

In [ ]:
brightness = images.mean(axis=(1, 2, 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [2, 1]})
sns.histplot(brightness, bins=40, ax=axes[0])
axes[0].set_title("Mean brightness per image")
axes[0].set_xlabel("Mean pixel value (0–1)")

axes[1].imshow(images.mean(axis=0))
axes[1].set_title("Average face")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 5. Save for the next notebooks

Saving the arrays means the next notebooks don't need to call `fetch_lfw_people` again and always use exactly the same data.

In [ ]:
np.savez_compressed(DATA_DIR / "lfw.npz", images=images, y=y, names=names)
counts.to_csv(DATA_DIR / "class_counts.csv", index=False)
print("Saved:", list(DATA_DIR.iterdir()))

## Observations (fill in after running)

- Number of people / images:
- How unbalanced is it? (largest class share, median images per person)
- Visible challenges (pose, lighting, glasses, …):
- Consequences for modelling: stratified train/test split, macro F1, `class_weight="balanced"`